### 2048 fps downsampling

In [ ]:
import os
import shutil
import pandas as pd
import torch
import sys
sys.path.append('/esail4/heeju/Point-M2AE/')
from utils.misc import fps
import glob

# 원본 폴더 경로
source_dir = '/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/Full_tree_for_svm_8192'
# 새 폴더 경로
target_dir = '/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/Full_tree_for_svm_2048'
sample_size = 2048

os.makedirs(target_dir, exist_ok=True)

# glob 함수 호출 시 glob.glob() 사용
csv_files = sorted(glob.glob(os.path.join(source_dir, '**', '*.csv'), recursive=True))

for i, file in enumerate(csv_files):
    print(f'{file}')
    df = pd.read_csv(file)
    lw_df = df[["X", "Y", "Z"]]

    lw_points = torch.tensor(lw_df.values).cuda().float().contiguous().unsqueeze(0)
    print(lw_points.shape)
    
    if len(lw_points[0]) >= sample_size:
        sampled_points = fps(lw_points, sample_size)
        sampled_points = sampled_points.cpu().numpy().squeeze()
        sampled_df = pd.DataFrame(sampled_points, columns=["X", "Y", "Z"])
    else:
        sampled_df = pd.DataFrame(lw_points.cpu().numpy().squeeze(), columns=["X", "Y", "Z"])  # 안전상 처리 (매우 적은 경우)
        
    print(sampled_df.shape)
    filename = f"{file}"
    save_path = os.path.join(target_dir, filename)
    print(save_path)
    sampled_df.to_csv(save_path, index=False)


In [ ]:
import os
import shutil
from glob import glob

# 원본 폴더 경로
source_folder = '/esail4/heeju/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/RW01_8192_LX'
# 새 폴더 경로
destination_folder = '/esail4/heeju/REGRESSION/M2AE_PRETRAIN/shapenet_rw1'

# 새 폴더가 존재하지 않으면 생성
os.makedirs(destination_folder, exist_ok=True)

# 파일을 새로 복사하고 이름 변경할 카운터 초기화
counter = 0

# 모든 하위 폴더에서 CSV 파일 경로를 가져오기
csv_files = sorted(glob(os.path.join(source_folder, '**', '*.csv'), recursive=True))

# CSV 파일들을 복사하고 이름을 변경
for file in csv_files:
    # 새 파일 이름
    new_name = f"99999999-{str(counter).zfill(8)}.csv"
    # 새 파일 경로
    destination_file = os.path.join(destination_folder, new_name)
    # 파일 복사
    shutil.copy2(file, destination_file)
    counter += 1

print("파일 복사가 완료되었습니다.")

In [ ]:
import os
import shutil
import pandas as pd
import torch
import sys
sys.path.append('/esail4/heeju/Point-M2AE/')
from utils.misc import fps
import glob

# 원본 폴더 경로
source_dir = '/esail4/heeju/REGRESSION/M2AE_PRETRAIN/shapenet_rw_for_test'
# 새 폴더 경로
target_dir = '/esail4/heeju/REGRESSION/M2AE_PRETRAIN/shapenet_rw_for_svm'
sample_size = 2048

os.makedirs(target_dir, exist_ok=True)

# glob 함수 호출 시 glob.glob() 사용
csv_files = sorted(glob.glob(os.path.join(source_dir, '**', '*.csv'), recursive=True))

for i, file in enumerate(csv_files):
    print(f'{file}')
    df = pd.read_csv(file)
    lw_df = df[["X", "Y", "Z"]]

    lw_points = torch.tensor(lw_df.values).cuda().float().contiguous().unsqueeze(0)
    print(lw_points.shape)
    
    if len(lw_points[0]) >= sample_size:
        sampled_points = fps(lw_points, sample_size)
        sampled_points = sampled_points.cpu().numpy().squeeze()
        sampled_df = pd.DataFrame(sampled_points, columns=["X", "Y", "Z"])
    else:
        sampled_df = pd.DataFrame(lw_points.cpu().numpy().squeeze(), columns=["X", "Y", "Z"])  # 안전상 처리 (매우 적은 경우)
        
    print(sampled_df.shape)
    filename = os.path.basename(file)
    save_path = os.path.join(target_dir, filename)
    print(save_path)
    sampled_df.to_csv(save_path, index=False)


In [ ]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#######################
# 0. 라벨 맵 정의 (예시)
#######################
# ModelNet40 클래스 + tree (추가)
modelnet40_classes = [
    "airplane", "bathtub", "bed", "bench", "bookshelf", "bottle", "bowl",
    "car", "chair", "cone", "cup", "curtain", "desk", "door", "dresser",
    "flower_pot", "glass_box", "guitar", "keyboard", "lamp", "laptop",
    "mantel", "monitor", "night_stand", "person", "piano", "plant", "radio",
    "range_hood", "sink", "sofa", "stairs", "stool", "table", "tent",
    "toilet", "tv_stand", "vase", "wardrobe", "xbox", "tree"
]
class2label = {c: i for i, c in enumerate(modelnet40_classes)}  # 예: airplane=0, ..., wood=40, leaf=41

###############################################
# 1. 기존 ModelNet40 HDF5 파일에서 data/label만 로드
###############################################
def load_modelnet_data(h5_dir, partition='train'):
    all_data = []
    all_label = []
    h5_files = glob.glob(os.path.join(h5_dir, f'ply_data_{partition}*.h5'))
    h5_files = sorted(h5_files)
    for h5_name in h5_files:
        with h5py.File(h5_name, 'r') as f:
            data = f['data'][:]   
            label = f['label'][:] 
        all_data.append(data)
        all_label.append(label)
    if len(all_data) == 0:
        return None, None
    all_data = np.concatenate(all_data, axis=0)
    all_label = np.concatenate(all_label, axis=0)
    if len(all_label.shape) == 2:
        all_label = all_label.squeeze()
    return all_data, all_label

##########################################
# 2. CSV 데이터 읽어오기 (헤더 없는 2048×3, wood/leaf 라벨 지정)
##########################################
def load_csv_data(csv_dir, category):
    csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))
    csv_data_list = []
    csv_label_list = []
    
    label_id = class2label[category]
    
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, header=None)  # 헤더를 무시하고 데이터를 읽음
        
        # 첫 번째 행이 'X', 'Y', 'Z'와 같은 문자열이 포함되어 있을 경우 제거
        df = df[1:]  # 첫 번째 행을 삭제 (첫 번째 행이 컬럼 이름일 경우)
        arr = df.values.astype(np.float32)  # 데이터를 float32로 변환
        
        # 데이터 형태가 (2048, 3)인지 확인
        if arr.shape != (2048, 3):
            print(f"Warning: {csv_file} shape {arr.shape}, expected (2048,3). Skip.")
            continue
        
        csv_data_list.append(arr)
        csv_label_list.append(label_id)
    
    if len(csv_data_list) == 0:
        return None, None
    csv_data = np.stack(csv_data_list, axis=0)
    csv_label = np.array(csv_label_list, dtype=np.int64)
    return csv_data, csv_label


###############################################
# 3. 기존 데이터와 CSV 데이터를 각각 합치기 (train/test 별도)
###############################################
def merge_datasets(modelnet_data, modelnet_label, csv_data_dict):
    csv_train_data_list, csv_test_data_list = [], []
    csv_train_label_list, csv_test_label_list = [], []
    
    for category, (csv_data, csv_label) in csv_data_dict.items():
        if csv_data is not None and csv_data.shape[0] > 0:
            csv_train_data, csv_test_data, csv_train_label, csv_test_label = train_test_split(
                csv_data, csv_label, test_size=0.2, random_state=42
            )
            csv_train_data_list.append(csv_train_data)
            csv_test_data_list.append(csv_test_data)
            csv_train_label_list.append(csv_train_label)
            csv_test_label_list.append(csv_test_label)
    
    train_data = np.concatenate([modelnet_data.get('train', np.empty((0, 2048, 3), dtype=np.float32))] + csv_train_data_list, axis=0)
    train_label = np.concatenate([modelnet_label.get('train', np.empty((0,), dtype=np.int64))] + csv_train_label_list, axis=0)
    test_data = np.concatenate([modelnet_data.get('test', np.empty((0, 2048, 3), dtype=np.float32))] + csv_test_data_list, axis=0)
    test_label = np.concatenate([modelnet_label.get('test', np.empty((0,), dtype=np.int64))] + csv_test_label_list, axis=0)
    
    return train_data, train_label, test_data, test_label

##########################################
# 4. HDF5 파일로 저장
##########################################
def save_h5_chunked(data, label, out_prefix, chunk_size=2048):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    N = data.shape[0]
    num_chunks = (N + chunk_size - 1) // chunk_size
    
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i+1) * chunk_size, N)
        chunk_data = data[start:end]
        chunk_label = label[start:end]
        filename = f"{out_prefix}_{i}.h5"
        with h5py.File(filename, 'w') as f:
            f.create_dataset('data', data=chunk_data, compression='gzip', compression_opts=4)
            f.create_dataset('label', data=chunk_label, compression='gzip', compression_opts=4)
        print(f"Saved {filename} with {chunk_data.shape[0]} samples")

##########################################
# 5. 메인 실행 예시
##########################################
if __name__ == "__main__":
    modelnet_h5_dir = "/esail4/heeju/M2AE_LW/data/pretrain/modelnet40_ply_hdf5_2048"
    csv_dirs = {
        "tree": "/esail4/heeju/REGRESSION/M2AE_PRETRAIN/shapenet_rw_for_svm",
    }
    out_dir = "/esail4/heeju/REGRESSION/M2AE_PRETRAIN/modelnet_tree_h5"
    os.makedirs(out_dir, exist_ok=True)
    
    # ModelNet 데이터 로드
    train_data_m, train_label_m = load_modelnet_data(modelnet_h5_dir, partition='train')
    test_data_m, test_label_m = load_modelnet_data(modelnet_h5_dir, partition='test')
    
    # CSV 데이터 로드
    csv_data_dict = {cat: load_csv_data(dir, cat) for cat, dir in csv_dirs.items()}
    
    # 기존 데이터와 CSV 데이터 합치기
    merged_train_data, merged_train_label, merged_test_data, merged_test_label = merge_datasets(
        {'train': train_data_m, 'test': test_data_m}, {'train': train_label_m, 'test': test_label_m}, csv_data_dict
    )
    
    # 합쳐진 데이터를 HDF5 파일로 저장
    save_h5_chunked(merged_train_data, merged_train_label, os.path.join(out_dir, "ply_data_train_tree"), chunk_size=2048)
    save_h5_chunked(merged_test_data, merged_test_label, os.path.join(out_dir, "ply_data_test_tree"), chunk_size=2048)


In [ ]:
import os
import glob
import shutil
import random

# 원본 폴더 경로
source_dir = '/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/shapenet_tree'
# 대상 폴더 경로 (test 데이터)
target_dir = '/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/Full_test_8192'

# 대상 폴더가 없으면 생성
os.makedirs(target_dir, exist_ok=True)

# 파일 목록 가져오기
csv_files = glob.glob(os.path.join(source_dir, "*.csv"))

# 파일을 무작위로 섞기
random.shuffle(csv_files)

# 30%만 선택하여 test 데이터로 이동
test_size = int(0.3 * len(csv_files))

# test 데이터를 target_dir로 이동
for i in range(test_size):
    file_to_move = csv_files[i]
    target_file = os.path.join(target_dir, os.path.basename(file_to_move))  # 파일 이름 유지

    # 파일 이동
    shutil.move(file_to_move, target_file)
    print(f"Moved {file_to_move} to {target_file}")

print(f"Completed moving {test_size} files to the test folder.")


### PLY Data Path set

In [ ]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

#######################
# 0. 라벨 맵 정의 (예시)
#######################
# ModelNet40 클래스 + tree (추가)
modelnet40_classes = [
    "airplane", "bathtub", "bed", "bench", "bookshelf", "bottle", "bowl",
    "car", "chair", "cone", "cup", "curtain", "desk", "door", "dresser",
    "flower_pot", "glass_box", "guitar", "keyboard", "lamp", "laptop",
    "mantel", "monitor", "night_stand", "person", "piano", "plant", "radio",
    "range_hood", "sink", "sofa", "stairs", "stool", "table", "tent",
    "toilet", "tv_stand", "vase", "wardrobe", "xbox", "tree"
]
class2label = {c: i for i, c in enumerate(modelnet40_classes)}  # 예: airplane=0, ..., wood=40, leaf=41

###############################################
# 1. 기존 ModelNet40 HDF5 파일에서 data/label만 로드
###############################################
def load_modelnet_data(h5_dir, partition='train'):
    all_data = []
    all_label = []
    h5_files = glob.glob(os.path.join(h5_dir, f'ply_data_{partition}*.h5'))
    h5_files = sorted(h5_files)
    for h5_name in h5_files:
        with h5py.File(h5_name, 'r') as f:
            data = f['data'][:]   
            label = f['label'][:] 
        all_data.append(data)
        all_label.append(label)
    if len(all_data) == 0:
        return None, None
    all_data = np.concatenate(all_data, axis=0)
    all_label = np.concatenate(all_label, axis=0)
    if len(all_label.shape) == 2:
        all_label = all_label.squeeze()
    return all_data, all_label

##########################################
# 2. CSV 데이터 읽어오기 (헤더 없는 2048×3, wood/leaf 라벨 지정)
##########################################
def load_csv_data(csv_dir, category):
    csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))
    csv_data_list = []
    csv_label_list = []
    
    label_id = class2label[category]
    
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, header=None)  # 헤더를 무시하고 데이터를 읽음
        
        # 첫 번째 행이 'X', 'Y', 'Z'와 같은 문자열이 포함되어 있을 경우 제거
        df = df[1:]  # 첫 번째 행을 삭제 (첫 번째 행이 컬럼 이름일 경우)
        arr = df.values.astype(np.float32)  # 데이터를 float32로 변환
        
        # 데이터 형태가 (2048, 3)인지 확인
        if arr.shape != (2048, 3):
            print(f"Warning: {csv_file} shape {arr.shape}, expected (2048,3). Skip.")
            continue
        
        csv_data_list.append(arr)
        csv_label_list.append(label_id)
    
    if len(csv_data_list) == 0:
        return None, None
    csv_data = np.stack(csv_data_list, axis=0)
    csv_label = np.array(csv_label_list, dtype=np.int64)
    return csv_data, csv_label


###############################################
# 3. 기존 데이터와 CSV 데이터를 각각 합치기 (train/test 별도)
###############################################
def merge_datasets(modelnet_data, modelnet_label, csv_data_dict):
    csv_train_data_list, csv_test_data_list = [], []
    csv_train_label_list, csv_test_label_list = [], []
    
    for category, (csv_data, csv_label) in csv_data_dict.items():
        if csv_data is not None and csv_data.shape[0] > 0:
            csv_train_data, csv_test_data, csv_train_label, csv_test_label = train_test_split(
                csv_data, csv_label, test_size=0.2, random_state=42
            )
            csv_train_data_list.append(csv_train_data)
            csv_test_data_list.append(csv_test_data)
            csv_train_label_list.append(csv_train_label)
            csv_test_label_list.append(csv_test_label)
    
    train_data = np.concatenate([modelnet_data.get('train', np.empty((0, 2048, 3), dtype=np.float32))] + csv_train_data_list, axis=0)
    train_label = np.concatenate([modelnet_label.get('train', np.empty((0,), dtype=np.int64))] + csv_train_label_list, axis=0)
    test_data = np.concatenate([modelnet_data.get('test', np.empty((0, 2048, 3), dtype=np.float32))] + csv_test_data_list, axis=0)
    test_label = np.concatenate([modelnet_label.get('test', np.empty((0,), dtype=np.int64))] + csv_test_label_list, axis=0)
    
    return train_data, train_label, test_data, test_label

##########################################
# 4. HDF5 파일로 저장
##########################################
def save_h5_chunked(data, label, out_prefix, chunk_size=2048):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    N = data.shape[0]
    num_chunks = (N + chunk_size - 1) // chunk_size
    
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i+1) * chunk_size, N)
        chunk_data = data[start:end]
        chunk_label = label[start:end]
        filename = f"{out_prefix}_{i}.h5"
        with h5py.File(filename, 'w') as f:
            f.create_dataset('data', data=chunk_data, compression='gzip', compression_opts=4)
            f.create_dataset('label', data=chunk_label, compression='gzip', compression_opts=4)
        print(f"Saved {filename} with {chunk_data.shape[0]} samples")

##########################################
# 5. 메인 실행 예시
##########################################
if __name__ == "__main__":
    modelnet_h5_dir = "/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/MODEL_NET/modelnet40_ply_hdf5_2048"
    csv_dirs = {
        "tree": "/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/Full_tree_for_svm_2048",
    }
    out_dir = "/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/modelnet40_tree_h5"
    os.makedirs(out_dir, exist_ok=True)
    
    # ModelNet 데이터 로드
    train_data_m, train_label_m = load_modelnet_data(modelnet_h5_dir, partition='train')
    test_data_m, test_label_m = load_modelnet_data(modelnet_h5_dir, partition='test')
    
    # CSV 데이터 로드
    csv_data_dict = {cat: load_csv_data(dir, cat) for cat, dir in csv_dirs.items()}
    
    # 기존 데이터와 CSV 데이터 합치기
    merged_train_data, merged_train_label, merged_test_data, merged_test_label = merge_datasets(
        {'train': train_data_m, 'test': test_data_m}, {'train': train_label_m, 'test': test_label_m}, csv_data_dict
    )
    
    # 합쳐진 데이터를 HDF5 파일로 저장
    save_h5_chunked(merged_train_data, merged_train_label, os.path.join(out_dir, "ply_data_train_tree"), chunk_size=2048)
    save_h5_chunked(merged_test_data, merged_test_label, os.path.join(out_dir, "ply_data_test_tree"), chunk_size=2048)


In [ ]:
import h5py

# HDF5 파일 경로
file_path = '/bess25/heeju/DATA/REGRESSION/M2AE_PRETRAIN/PROCESSED/DOWNSAMPLED/modelnet40_tree_h5/ply_data_train_tree_0.h5'

# HDF5 파일 열기
with h5py.File(file_path, 'r') as f:
    # 파일 내의 모든 그룹과 데이터셋 확인
    print("Keys in the file:", list(f.keys()))  # 파일 내의 모든 그룹 이름 출력
    
    # 파일 내의 특정 데이터셋에 대한 정보 확인 (예: 'data'와 'label'이 있을 것)
    if 'data' in f:
        print("\nData Shape:", f['data'].shape)  # 'data' 데이터셋의 shape 출력
        print("Data Sample:", f['data'][:5])  # 데이터의 처음 5개 샘플 출력

    if 'label' in f:
        print("\nLabel Shape:", f['label'].shape)  # 'label' 데이터셋의 shape 출력
        print("Label Sample:", f['label'][:5])  # 레이블의 처음 5개 샘플 출력
